# vsgame_exp02 Statistics
## Agency broadly enhances memory beyond directly controlled information

## Set Up

### Import Packages

In [1]:
import matplotlib.pyplot as plt
import glob
import os
import pandas as pd
import re
import seaborn as sns
import itertools
import statsmodels.api as sm 
import statsmodels.formula.api as smf
# import ptitprince as pt

### Load Data

In [12]:
base_dir = '/Users/aidelarazan/Box Sync/aidelarazan_box/Projects/vsgame/exp02/data/'
mst = pd.read_csv(os.path.join(base_dir, 'vsgame_exp02_sub-all_task-MST.csv'))
spatial_placement = pd.read_csv(os.path.join(base_dir, 'vsgame_exp02_sub-all_task-SpatialPlacement_desc-Accuracy.csv'))
item_desc = pd.read_csv(os.path.join(base_dir, 'vsgame_exp02_sub-all_task-ItemRecallDescriptions_desc-SemanticSimilarity.csv'))
spatial_desc = pd.read_csv(os.path.join(base_dir, 'vsgame_exp02_sub-all_task-SpatialRecallDescriptions_desc-SemanticSimilarity.csv'))
attention = pd.read_csv(os.path.join(base_dir, 'vsgame_exp02_sub-all_task-AttentionRatings.csv'))
questionnaire = pd.read_csv(os.path.join(base_dir, 'vsgame_exp02_sub-all_task-PostTaskSurvey.csv'))
demographics = pd.read_csv(os.path.join(base_dir, 'vsgame_exp02_sub-all_task-Demographics.csv'))

In [13]:
attention = attention.rename(columns={'mean_rating':'attention_rating'})
attention = attention[['subject', 'condition', 'attention_rating']].reset_index(drop=True)
attention


,subject,condition,attention_rating
0,103,Active,2.666667
1,103,Passive,2.666667
2,109,Active,2.000000
3,109,Passive,4.000000
4,113,Active,3.000000
...,...,...,...
57,8,Passive,3.333333
58,9,Active,2.333333
59,9,Passive,6.000000
60,92,Active,1.333333


## MST

In [14]:
mst.head()

,subject,experiment,task,condition,target_hit,target_miss,lure_cr,lure_fa,foil_cr,foil_fa,...,fa,fa_rate,dprime,dprime_lures,LDI,beta,c,ad,foiled_target,foiled_lure
0,2,exp02,mst,Active,2,1,6,2,7,2,...,4,0.222222,0.000000,0.000000,0.555556,1.000000,0.764710,0.500000,1,1
1,2,exp02,mst,Passive,0,9,0,0,9,0,...,0,0.000000,0.000000,0.000000,-1.000000,inf,inf,0.500000,9,9
2,5,exp02,mst,Active,1,3,5,3,8,1,...,4,0.222222,0.000000,-0.789913,0.222222,0.635978,0.992675,0.500000,3,1
3,5,exp02,mst,Passive,6,1,5,0,8,1,...,1,0.055556,1.651368,2.023946,0.444444,3.242712,0.581246,0.878535,1,4
4,6,exp02,mst,Active,3,0,2,4,8,1,...,5,0.277778,0.789913,-0.291017,0.222222,1.084334,0.510092,0.711767,0,3


In [15]:
mst = pd.merge(mst, attention, on=['subject', 'condition'], how='left')
mst

,subject,experiment,task,condition,target_hit,target_miss,lure_cr,lure_fa,foil_cr,foil_fa,...,fa_rate,dprime,dprime_lures,LDI,beta,c,ad,foiled_target,foiled_lure,attention_rating
0,2,exp02,mst,Active,2,1,6,2,7,2,...,0.222222,0.000000,0.000000,0.555556,1.000000,0.764710,0.500000,1,1,1.666667
1,2,exp02,mst,Passive,0,9,0,0,9,0,...,0.000000,0.000000,0.000000,-1.000000,inf,inf,0.500000,9,9,6.000000
2,5,exp02,mst,Active,1,3,5,3,8,1,...,0.222222,0.000000,-0.789913,0.222222,0.635978,0.992675,0.500000,3,1,0.000000
3,5,exp02,mst,Passive,6,1,5,0,8,1,...,0.055556,1.651368,2.023946,0.444444,3.242712,0.581246,0.878535,1,4,3.333333
4,6,exp02,mst,Active,3,0,2,4,8,1,...,0.277778,0.789913,-0.291017,0.222222,1.084334,0.510092,0.711767,0,3,1.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,126,exp02,mst,Passive,2,6,0,1,9,0,...,0.055556,0.828509,0.455931,-0.666667,2.655898,1.178964,0.721010,6,8,5.666667
58,128,exp02,mst,Active,9,0,2,6,7,2,...,0.444444,2.357928,1.162492,0.222222,0.283820,-0.726754,0.952273,0,1,0.000000
59,128,exp02,mst,Passive,7,1,1,5,6,3,...,0.444444,1.195437,0.624999,0.000000,0.753798,-0.312500,0.801029,1,3,1.666667
60,131,exp02,mst,Active,6,0,1,7,5,4,...,0.611111,0.570438,-0.333982,0.111111,0.948437,-0.356472,0.656658,0,1,0.000000


### dprime

In [16]:
mst.groupby('condition')['dprime'].describe().reset_index()

,condition,count,mean,std,min,25%,50%,75%,max
0,Active,31.0,0.853575,0.950775,-1.360351,0.291017,0.789913,1.529419,2.813859
1,Passive,31.0,0.182410,0.844935,-1.529419,-0.312500,0.279421,0.597718,1.985350


In [17]:
model = smf.mixedlm(
    data = mst, 
    formula = "dprime ~ condition", 
    groups = mst['subject'],
)
result = model.fit(reml=True)

print(result.summary())


             Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    dprime  
No. Observations:    62         Method:                REML    
No. Groups:          31         Scale:                 0.6220  
Min. group size:     2          Log-Likelihood:        -81.3865
Max. group size:     2          Converged:             Yes     
Mean group size:     2.0                                       
---------------------------------------------------------------
                     Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------
Intercept             0.854    0.162  5.284 0.000  0.537  1.170
condition[T.Passive] -0.671    0.200 -3.350 0.001 -1.064 -0.279
Group Var             0.187    0.231                           



In [18]:
model = smf.mixedlm(
    data = mst, 
    formula = "dprime ~ condition + attention_rating", 
    groups = mst['subject'],
)
result = model.fit(reml=True)

print(result.summary())


             Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    dprime  
No. Observations:    62         Method:                REML    
No. Groups:          31         Scale:                 0.6100  
Min. group size:     2          Log-Likelihood:        -82.3060
Max. group size:     2          Converged:             Yes     
Mean group size:     2.0                                       
---------------------------------------------------------------
                     Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------
Intercept             0.750    0.186  4.033 0.000  0.386  1.115
condition[T.Passive] -0.840    0.249 -3.373 0.001 -1.329 -0.352
attention_rating      0.095    0.085  1.123 0.262 -0.071  0.262
Group Var             0.198    0.236                           



### LDI

In [19]:
mst.groupby('condition')['LDI'].describe()

,count,mean,std,min,25%,50%,75%,max
condition,,,,,,,,
Active,31.0,0.143369,0.215058,-0.333333,0.055556,0.111111,0.222222,0.666667
Passive,31.0,-0.172043,0.362704,-1.000000,-0.444444,-0.111111,0.055556,0.555556


In [20]:
model = smf.mixedlm(
    data = mst, 
    formula = "LDI ~ condition", 
    groups = mst['subject'],
)
result = model.fit(reml=True)

print(result.summary())


             Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    LDI     
No. Observations:    62         Method:                REML    
No. Groups:          31         Scale:                 0.0511  
Min. group size:     2          Log-Likelihood:        -12.9668
Max. group size:     2          Converged:             Yes     
Mean group size:     2.0                                       
---------------------------------------------------------------
                     Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------
Intercept             0.143    0.054  2.677 0.007  0.038  0.248
condition[T.Passive] -0.315    0.057 -5.495 0.000 -0.428 -0.203
Group Var             0.038    0.102                           



In [21]:
model = smf.mixedlm(
    data = mst, 
    formula = "LDI ~ condition + attention_rating", 
    groups = mst['subject'],
)
result = model.fit(reml=True)

print(result.summary())


             Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    LDI     
No. Observations:    62         Method:                REML    
No. Groups:          31         Scale:                 0.0505  
Min. group size:     2          Log-Likelihood:        -15.3512
Max. group size:     2          Converged:             Yes     
Mean group size:     2.0                                       
---------------------------------------------------------------
                     Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------
Intercept             0.166    0.062  2.687 0.007  0.045  0.288
condition[T.Passive] -0.278    0.076 -3.681 0.000 -0.426 -0.130
attention_rating     -0.021    0.028 -0.756 0.450 -0.076  0.034
Group Var             0.040    0.107                           



## Spatial Placement

In [22]:
spatial_placement.head()

,subject,task,experiment,condition,correct
0,2,spatial_placement,exp02,Active,0.277778
1,2,spatial_placement,exp02,Passive,0.277778
2,5,spatial_placement,exp02,Active,0.222222
3,5,spatial_placement,exp02,Passive,0.333333
4,6,spatial_placement,exp02,Active,0.500000


In [23]:
spatial_placement = pd.merge(spatial_placement, attention, on=['subject', 'condition'], how='left')
spatial_placement

,subject,task,experiment,condition,correct,attention_rating
0,2,spatial_placement,exp02,Active,0.277778,1.666667
1,2,spatial_placement,exp02,Passive,0.277778,6.000000
2,5,spatial_placement,exp02,Active,0.222222,0.000000
3,5,spatial_placement,exp02,Passive,0.333333,3.333333
4,6,spatial_placement,exp02,Active,0.500000,1.333333
...,...,...,...,...,...,...
57,126,spatial_placement,exp02,Passive,0.166667,5.666667
58,128,spatial_placement,exp02,Active,0.555556,0.000000
59,128,spatial_placement,exp02,Passive,0.666667,1.666667
60,131,spatial_placement,exp02,Active,0.333333,0.000000


In [24]:
spatial_placement.groupby('condition')['correct'].describe()

,count,mean,std,min,25%,50%,75%,max
condition,,,,,,,,
Active,31.0,0.569892,0.225881,0.222222,0.388889,0.555556,0.750000,0.944444
Passive,31.0,0.370968,0.160623,0.055556,0.277778,0.388889,0.444444,0.833333


In [25]:
model = smf.mixedlm(
    data = spatial_placement, 
    formula = "correct ~ condition", 
    groups = spatial_placement['subject'],
)
result = model.fit(reml=True)

print(result.summary())


             Mixed Linear Model Regression Results
Model:                MixedLM    Dependent Variable:    correct
No. Observations:     62         Method:                REML   
No. Groups:           31         Scale:                 0.0294 
Min. group size:      2          Log-Likelihood:        10.0610
Max. group size:      2          Converged:             Yes    
Mean group size:      2.0                                      
---------------------------------------------------------------
                     Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------
Intercept             0.570    0.035 16.190 0.000  0.501  0.639
condition[T.Passive] -0.199    0.044 -4.567 0.000 -0.284 -0.114
Group Var             0.009    0.050                           



/Users/aidelarazan/miniconda3/envs/psifr/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [26]:
model = smf.mixedlm(
    data = spatial_placement, 
    formula = "correct ~ condition + attention_rating", 
    groups = spatial_placement['subject'],
)
result = model.fit(reml=True)

print(result.summary())


             Mixed Linear Model Regression Results
Model:                MixedLM    Dependent Variable:    correct
No. Observations:     62         Method:                REML   
No. Groups:           31         Scale:                 0.0304 
Min. group size:      2          Log-Likelihood:        7.2947 
Max. group size:      2          Converged:             Yes    
Mean group size:      2.0                                      
---------------------------------------------------------------
                     Coef.  Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------
Intercept             0.554    0.041 13.609 0.000  0.474  0.634
condition[T.Passive] -0.225    0.055 -4.060 0.000 -0.334 -0.116
attention_rating      0.015    0.019  0.783 0.434 -0.022  0.052
Group Var             0.008    0.050                           



/Users/aidelarazan/miniconda3/envs/psifr/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


### Testing Against Chance

In [27]:
from scipy import stats

chance = 1/3  # three location markers

for cond in ["Active", "Passive"]:
    vals = spatial_placement.loc[spatial_placement["condition"] == cond, "correct"]
    t, p = stats.ttest_1samp(vals, popmean=chance)
    print(f"{cond}: M = {vals.mean():.3f}, SD = {vals.std():.3f}, "
          f"n = {vals.count()}, t({vals.count()-1}) = {t:.3f}, p = {p:.4f}")

Active: M = 0.570, SD = 0.226, n = 31, t(30) = 5.831, p = 0.0000
Passive: M = 0.371, SD = 0.161, n = 31, t(30) = 1.305, p = 0.2020


## Item Recall Description

In [28]:
item_desc = item_desc.groupby(['subject', 'condition', 'experiment']).mean('semantic_similarity').reset_index()
item_desc

,subject,condition,experiment,semantic_similarity
0,2,Active,exp02,0.320045
1,2,Passive,exp02,0.025534
2,5,Active,exp02,0.364013
3,5,Passive,exp02,0.429321
4,6,Active,exp02,0.255692
...,...,...,...,...
57,126,Passive,exp02,0.088483
58,128,Active,exp02,0.412110
59,128,Passive,exp02,0.338884
60,131,Active,exp02,0.183449


In [29]:
item_desc = pd.merge(item_desc, attention, on=['subject', 'condition'], how='left')
item_desc

,subject,condition,experiment,semantic_similarity,attention_rating
0,2,Active,exp02,0.320045,1.666667
1,2,Passive,exp02,0.025534,6.000000
2,5,Active,exp02,0.364013,0.000000
3,5,Passive,exp02,0.429321,3.333333
4,6,Active,exp02,0.255692,1.333333
...,...,...,...,...,...
57,126,Passive,exp02,0.088483,5.666667
58,128,Active,exp02,0.412110,0.000000
59,128,Passive,exp02,0.338884,1.666667
60,131,Active,exp02,0.183449,0.000000


In [30]:
item_desc.groupby('condition')['semantic_similarity'].describe()

,count,mean,std,min,25%,50%,75%,max
condition,,,,,,,,
Active,31.0,0.270721,0.125212,0.020836,0.178273,0.273599,0.360861,0.469339
Passive,31.0,0.171185,0.123047,-0.022450,0.093044,0.149882,0.245115,0.472453


In [31]:
model = smf.mixedlm(
    data = item_desc, 
    formula = "semantic_similarity ~ condition", 
    groups = item_desc['subject'],
)
result = model.fit(reml=True)

print(result.summary())


              Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: semantic_similarity
No. Observations: 62      Method:             REML               
No. Groups:       31      Scale:              0.0054             
Min. group size:  2       Log-Likelihood:     44.8397            
Max. group size:  2       Converged:          Yes                
Mean group size:  2.0                                            
-----------------------------------------------------------------
                       Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------
Intercept               0.271    0.022 12.143 0.000  0.227  0.314
condition[T.Passive]   -0.100    0.019 -5.334 0.000 -0.136 -0.063
Group Var               0.010    0.063                           



In [32]:
model = smf.mixedlm(
    data = item_desc, 
    formula = "semantic_similarity ~ condition + attention_rating", 
    groups = item_desc['subject'],
)
result = model.fit(reml=True)

print(result.summary())


              Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: semantic_similarity
No. Observations: 62      Method:             REML               
No. Groups:       31      Scale:              0.0055             
Min. group size:  2       Log-Likelihood:     41.2051            
Max. group size:  2       Converged:          Yes                
Mean group size:  2.0                                            
-----------------------------------------------------------------
                       Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------
Intercept               0.273    0.025 10.774 0.000  0.223  0.322
condition[T.Passive]   -0.096    0.027 -3.603 0.000 -0.148 -0.044
attention_rating       -0.002    0.011 -0.178 0.859 -0.023  0.019
Group Var               0.010    0.066                           



## Spatial Recall Description

In [33]:
spatial_desc

,subject,condition,room,recall,original_description,semantic_similarity,task,experiment
0,103,Passive,Arcade,a room where there no games,"In this room, there is a leaderboard that upda...",0.224862,recall,exp02
1,103,Active,Laundry Room,a room where laundry gets done by itself but n...,"In this room, the clothes get washed, dried, a...",0.485934,recall,exp02
2,103,Passive,Theater,no lights,"In this room, a movie plays automatically and ...",0.014871,recall,exp02
3,103,Active,Swimming Pool,water stays still not matter what,"In this room, the water remains still, no matt...",0.520005,recall,exp02
4,103,Active,Bedroom,light hits your eyes,"In this room, the sun always hits your eyes in...",0.498143,recall,exp02
...,...,...,...,...,...,...,...,...
739,92,Active,Bowling Alley,bowling ball comes back automatically,"In this room, the bowling balls always roll ba...",0.519136,recall,exp02
740,92,Passive,Zen Room,NaN,"In this room, no sound can enter the walls.",-0.044518,recall,exp02
741,92,Active,Rooftop,lights up at night,"In this room, the solar panels light up at night.",0.511866,recall,exp02
742,92,Passive,Garage,can fit any amount of cars,"In this room, the walls will expand to fit any...",0.465221,recall,exp02


In [34]:
spatial_desc = spatial_desc.groupby(['subject', 'condition', 'experiment']).mean('semantic_similarity').reset_index()
spatial_desc

,subject,condition,experiment,semantic_similarity
0,2,Active,exp02,0.274646
1,2,Passive,exp02,0.102274
2,5,Active,exp02,0.502074
3,5,Passive,exp02,0.449597
4,6,Active,exp02,0.433456
...,...,...,...,...
57,126,Passive,exp02,0.034669
58,128,Active,exp02,0.542795
59,128,Passive,exp02,0.155445
60,131,Active,exp02,0.299194


In [35]:
spatial_desc = pd.merge(spatial_desc, attention, on=['subject', 'condition'], how='left')
spatial_desc

,subject,condition,experiment,semantic_similarity,attention_rating
0,2,Active,exp02,0.274646,1.666667
1,2,Passive,exp02,0.102274,6.000000
2,5,Active,exp02,0.502074,0.000000
3,5,Passive,exp02,0.449597,3.333333
4,6,Active,exp02,0.433456,1.333333
...,...,...,...,...,...
57,126,Passive,exp02,0.034669,5.666667
58,128,Active,exp02,0.542795,0.000000
59,128,Passive,exp02,0.155445,1.666667
60,131,Active,exp02,0.299194,0.000000


In [36]:
spatial_desc.groupby('condition')['semantic_similarity'].describe()

,count,mean,std,min,25%,50%,75%,max
condition,,,,,,,,
Active,31.0,0.324285,0.171919,0.018999,0.187957,0.301795,0.455228,0.633116
Passive,31.0,0.176064,0.125098,-0.004082,0.106905,0.164864,0.211819,0.546124


In [37]:
model = smf.mixedlm(
    data = spatial_desc, 
    formula = "semantic_similarity ~ condition", 
    groups = spatial_desc['subject'],
)
result = model.fit(reml=True)

print(result.summary())


              Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: semantic_similarity
No. Observations: 62      Method:             REML               
No. Groups:       31      Scale:              0.0081             
Min. group size:  2       Log-Likelihood:     33.0801            
Max. group size:  2       Converged:          Yes                
Mean group size:  2.0                                            
-----------------------------------------------------------------
                       Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------
Intercept               0.324    0.027 12.010 0.000  0.271  0.377
condition[T.Passive]   -0.148    0.023 -6.484 0.000 -0.193 -0.103
Group Var               0.015    0.075                           



In [38]:
model = smf.mixedlm(
    data = spatial_desc, 
    formula = "semantic_similarity ~ condition + attention_rating", 
    groups = spatial_desc['subject'],
)
result = model.fit(reml=True)

print(result.summary())


              Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: semantic_similarity
No. Observations: 62      Method:             REML               
No. Groups:       31      Scale:              0.0080             
Min. group size:  2       Log-Likelihood:     29.8587            
Max. group size:  2       Converged:          Yes                
Mean group size:  2.0                                            
-----------------------------------------------------------------
                       Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------
Intercept               0.334    0.031 10.890 0.000  0.274  0.394
condition[T.Passive]   -0.133    0.032 -4.128 0.000 -0.196 -0.070
attention_rating       -0.009    0.013 -0.684 0.494 -0.034  0.016
Group Var               0.015    0.079                           



## Post-Experiment Survey

In [39]:
questionnaire.head()

,subject,condition,experiment,task,question,response
0,2,Active,exp02,posttask_survey,memory,5
1,2,Passive,exp02,posttask_survey,memory,6
2,2,Active,exp02,posttask_survey,control,0
3,2,Passive,exp02,posttask_survey,control,6
4,5,Active,exp02,posttask_survey,memory,1


In [40]:
questionnaire.groupby(['question', 'condition'])['response'].agg(['mean', 'std', 'count']).round(2).reset_index()

,question,condition,mean,std,count
0,control,Active,1.94,1.93,31
1,control,Passive,4.35,2.07,31
2,memory,Active,2.71,1.66,31
3,memory,Passive,4.13,1.36,31


In [41]:
from scipy import stats

for q, g in questionnaire.groupby('question'):
    a = g.loc[g.condition == 'Active', 'response'].dropna()
    p = g.loc[g.condition == 'Passive', 'response'].dropna()
    t, pval = stats.ttest_ind(a, p)
    print(q, round(t, 2), round(pval, 4))

control -4.75 0.0
memory -3.69 0.0005


## Demographics

In [44]:
demographics

,Unnamed: 0,Study,Experiment,Subject,Age,Sex,Ethnicity,Race
0,0,vsgame,exp02,2,31,Male,Not Hispanic or Latino,White or Caucasian
1,1,vsgame,exp02,5,32,Male,Not Hispanic or Latino,White or Caucasian
2,2,vsgame,exp02,6,27,Female,Not Hispanic or Latino,White or Caucasian
3,3,vsgame,exp02,7,22,Female,Not Hispanic or Latino,Black or African American
4,4,vsgame,exp02,8,24,Female,Not Hispanic or Latino,Black or African American
5,5,vsgame,exp02,9,27,Male,Not Hispanic or Latino,White or Caucasian
6,6,vsgame,exp02,13,34,Male,Hispanic or Latino,White or Caucasian
7,7,vsgame,exp02,14,32,Prefer Not to Report,Not Hispanic or Latino,Prefer not to Report
8,8,vsgame,exp02,18,26,Female,Not Hispanic or Latino,Asian
9,9,vsgame,exp02,19,27,Female,Not Hispanic or Latino,Black or African American


In [42]:
demographics['Age'].describe().reset_index()

,index,Age
0,count,31.000000
1,mean,27.967742
2,std,4.460821
3,min,20.000000
4,25%,24.500000
5,50%,28.000000
6,75%,31.500000
7,max,34.000000


In [43]:
demographics['Sex'].value_counts()

Sex
Female                  17
Male                    13
Prefer Not to Report     1
Name: count, dtype: int64

In [45]:
demographics['Race'].value_counts()

Race
White or Caucasian                                      17
Black or African American                                5
Asian                                                    4
Other                                                    2
Prefer not to Report                                     1
Asian; White or Caucasian                                1
American Indian or Alaska Native; White or Caucasian     1
Name: count, dtype: int64